# Brahmi Embedding Warmup — Aggressive Round 2

**Goal**: Train the 300 new Devanagari embeddings using ALL 435K training rows.
Round 1 (150 steps, 2000 chunks) produced garbled Hindi. This round uses:
- Full corpus (435K rows → ~200K+ chunks)
- Higher LR (1e-3)
- Cosine schedule
- Only chunks containing new tokens (efficient training)

**Hardware**: T4 GPU (Colab free tier). ~2-3 hours.

**Before starting**: Paste your HF_TOKEN below.

In [ ]:
!pip install -q transformers torch datasets accelerate huggingface_hub

import json, os, math, time, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_TOKEN = "hf_YOUR_TOKEN_HERE"

# Mount Google Drive for persistence across sessions
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/brahmi_training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Saving to {DRIVE_DIR}')

from huggingface_hub import login
login(token=HF_TOKEN)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
MODEL_ID = 'eulogik/Bharat-Tiny-LLM-v2'

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
)

new_ids = sorted(
    tid for tid, t in tok.added_tokens_decoder.items()
    if not str(t).startswith('<') and not getattr(t, 'special', False)
)
print(f'Found {len(new_ids)} new tokens (IDs {new_ids[0]}–{new_ids[-1]})')

In [ ]:
for p in model.parameters():
    p.requires_grad = False
model.model.embed_tokens.weight.requires_grad_(True)
model.lm_head.weight.requires_grad_(True)

class LearnableRows(nn.Module):
    def __init__(self, embed, lmhead):
        super().__init__()
        self.embed = nn.Parameter(embed)
        self.lmhead = nn.Parameter(lmhead)

hidden = model.config.hidden_size
embed_w = model.model.embed_tokens.weight
lmhead_w = model.lm_head.weight

learner = LearnableRows(
    embed_w.data[new_ids].clone(),
    lmhead_w.data[new_ids].clone()
).to(device)

n_params = len(new_ids) * hidden * 2
print(f'Training {n_params:,} params (0.04% of {sum(p.numel() for p in model.parameters()):,})')

In [ ]:
import requests, gzip, shutil

DATA_URL = 'https://huggingface.co/eulogik/Bharat-Tiny-LLM-v2/resolve/main/train_gold_v3.jsonl.gz'
DATA_FILE = '/content/train_gold_v3.jsonl'

if not os.path.exists(DATA_FILE):
    print('Downloading...')
    r = requests.get(DATA_URL, stream=True,
                     headers={'Authorization': f'Bearer {HF_TOKEN}'})
    r.raise_for_status()
    with open('/content/data.gz', 'wb') as f:
        shutil.copyfileobj(r.raw, f)
    with gzip.open('/content/data.gz', 'rb') as gz, open(DATA_FILE, 'wb') as f:
        shutil.copyfileobj(gz, f)
    os.remove('/content/data.gz')
    print(f'Ready! {os.path.getsize(DATA_FILE) / 1e6:.0f} MB')

from datasets import load_dataset
dataset = load_dataset('text', data_files=DATA_FILE, split='train')
print(f'Loaded {len(dataset)} rows')

In [ ]:
CHUNKS_FILE = os.path.join(DRIVE_DIR, 'all_chunks.json')

if os.path.exists(CHUNKS_FILE):
    print('Loading cached chunks from Drive...')
    with open(CHUNKS_FILE) as f:
        all_chunks = json.load(f)
    random.shuffle(all_chunks)
    print(f'Loaded {len(all_chunks)} chunks')
else:
    new_id_set = set(new_ids)
    all_chunks = []
    skipped = 0

    for i, example in enumerate(dataset):
        text = example.get('text', '')
        if isinstance(text, str) and text.startswith('{'):
            try:
                data = json.loads(text)
                text = ' '.join(m['content'] for m in data.get('messages', []))
            except:
                pass
        ids = tok.encode(text)
        for j in range(0, len(ids), 256):
            chunk = ids[j:j+256]
            if len(chunk) >= 10:
                if any(tid in new_id_set for tid in chunk):
                    all_chunks.append(chunk)
                else:
                    skipped += 1
        if i % 50000 == 0 and i > 0:
            print(f'  Processed {i}/{len(dataset)} rows...')

    random.shuffle(all_chunks)
    with open(CHUNKS_FILE, 'w') as f:
        json.dump(all_chunks, f)
    print(f'\nKept {len(all_chunks)} chunks (skipped {skipped})')
    print(f'Saved to Drive')

In [ ]:
def collate(batch):
    mx = max(len(x) for x in batch)
    pad = torch.zeros(len(batch), mx, dtype=torch.long)
    for i, x in enumerate(batch):
        pad[i, :len(x)] = torch.tensor(x, dtype=torch.long)
    return pad.to(device)

split = int(len(all_chunks) * 0.95)
train_chunks = all_chunks[:split]
val_chunks = all_chunks[split:]

BATCH_SIZE = 4
train_loader = DataLoader(train_chunks, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_chunks, batch_size=BATCH_SIZE, collate_fn=collate)

print(f'Train: {len(train_chunks)} chunks ({len(train_loader)} batches)')
print(f'Val: {len(val_chunks)} chunks ({len(val_loader)} batches)')

In [ ]:
NUM_STEPS = 3000
LR = 1e-3
CKPT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

optim = torch.optim.AdamW(learner.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, T_max=NUM_STEPS, eta_min=1e-5
)

# Resume from checkpoint if exists
step = 0
best_loss = float('inf')
init_loss = None
resume_path = os.path.join(CKPT_DIR, 'state.json')
learner_path = os.path.join(CKPT_DIR, 'learner.pt')

if os.path.exists(resume_path) and os.path.exists(learner_path):
    with open(resume_path) as f:
        state = json.load(f)
    step = state['step']
    best_loss = state['best_loss']
    init_loss = state['init_loss']
    learner.load_state_dict(torch.load(learner_path, map_location=device))
    # Fast-forward scheduler
    for _ in range(step):
        scheduler.step()
    print(f'Resumed from step {step}, best loss {best_loss:.4f}')
else:
    # Initial validation loss
    model.eval()
    init_losses = []
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            init_losses.append(model(batch, labels=batch).loss.item())
            if i >= 20: break
    init_loss = np.mean(init_losses)
    best_loss = init_loss
    print(f'Initial val loss: {init_loss:.4f} (PPL: {math.exp(init_loss):.1f})')

In [ ]:
model.train()
start_t = time.time()
log_every = 100

for epoch in range(10):
    for batch in train_loader:
        if step >= NUM_STEPS:
            break

        # Copy learner → model
        with torch.no_grad():
            embed_w.data[new_ids] = learner.embed.data.to(embed_w.device)
            lmhead_w.data[new_ids] = learner.lmhead.data.to(lmhead_w.device)

        out = model(batch, labels=batch)
        loss = out.loss
        loss.backward()

        # Copy grads → learner
        with torch.no_grad():
            learner.embed.grad = embed_w.grad[new_ids].clone().to(learner.embed.device)
            learner.lmhead.grad = lmhead_w.grad[new_ids].clone().to(learner.lmhead.device)

        torch.nn.utils.clip_grad_norm_(learner.parameters(), 1.0)
        optim.step()
        scheduler.step()
        optim.zero_grad()
        model.zero_grad()
        step += 1

        if step % log_every == 0:
            model.eval()
            vlosses = []
            with torch.no_grad():
                for vb in val_loader:
                    vlosses.append(model(vb, labels=vb).loss.item())
                    if len(vlosses) >= 20: break
            avg_vloss = np.mean(vlosses)
            impr = (init_loss - avg_vloss) / init_loss * 100
            lr_now = scheduler.get_last_lr()[0]
            elapsed = time.time() - start_t
            print(f'Step {step}/{NUM_STEPS} | '
                  f'Tr:{loss.item():.4f} | '
                  f'Val:{avg_vloss:.4f} (PPL:{math.exp(avg_vloss):.1f}) | '
                  f'{impr:+.1f}% | '
                  f'LR:{lr_now:.2e} | '
                  f'{step/elapsed:.2f} it/s')
            if avg_vloss < best_loss:
                best_loss = avg_vloss
                with torch.no_grad():
                    embed_w.data[new_ids] = learner.embed.data.to(embed_w.device)
                    lmhead_w.data[new_ids] = learner.lmhead.data.to(lmhead_w.device)
                model.save_pretrained('brahmi-best')
                tok.save_pretrained('brahmi-best')
            # Save checkpoint for resume
            torch.save(learner.state_dict(), learner_path)
            with open(resume_path, 'w') as f:
                json.dump({'step': step, 'best_loss': best_loss, 'init_loss': init_loss}, f)
            model.train()

    if step >= NUM_STEPS:
        break

elapsed = time.time() - start_t
final_impr = (init_loss - best_loss) / init_loss * 100
print(f'\nDone! {step} steps in {elapsed:.0f}s ({step/elapsed:.2f} it/s)')
print(f'Init loss: {init_loss:.4f} -> Best: {best_loss:.4f} ({final_impr:.1f}% improvement)')
print(f'Init PPL: {math.exp(init_loss):.1f} -> Best PPL: {math.exp(best_loss):.1f}')

In [ ]:
prompts = [
    'मैं आपको बताना चाहता हूँ कि',
    'भारत की राजधानी',
    'नमस्ते, आप कैसे हैं? मैं',
]

for p in prompts:
    inputs = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.3,
        top_p=0.85,
        repetition_penalty=1.25,
        do_sample=True,
    )
    gen = tok.decode(out[0][inputs.input_ids.shape[-1]:],
                     skip_special_tokens=True)
    print(f'Prompt: {p}')
    print(f'  -> {gen}')
    print()

In [ ]:
from huggingface_hub import HfApi

with torch.no_grad():
    embed_w.data[new_ids] = learner.embed.data.to(embed_w.device)
    lmhead_w.data[new_ids] = learner.lmhead.data.to(lmhead_w.device)

model.save_pretrained('brahmi-trained')
tok.save_pretrained('brahmi-trained')
print('Saved locally')

api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path='brahmi-trained',
    repo_id='eulogik/Bharat-Tiny-LLM-v2',
    repo_type='model',
    commit_message=f'continued_pretrain_round2: {final_impr:.1f}% val loss improvement',
)
print('Uploaded to HF!')
print('https://huggingface.co/eulogik/Bharat-Tiny-LLM-v2')